# 02 · Farm advisor with a tool (function calling)
**Time:** 15 min · **Goal:** let the model look up facts from your own function instead of guessing.

Run the cells from top to bottom. **Do not share your key and do not upload private data.** Free-tier content may be used by Google to improve its products.

Get a key: https://aistudio.google.com/apikey

In [ ]:
# Setup: install the Gemini SDK and load your free AI Studio key
!pip -q install google-genai
import os, getpass
try:
    from google.colab import userdata          # Colab: left menu > Secrets > add GOOGLE_API_KEY
    API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    API_KEY = os.environ.get("GOOGLE_API_KEY") or getpass.getpass("Paste your AI Studio key: ")
from google import genai
from google.genai import types
client = genai.Client(api_key=API_KEY)
# Model names retire quickly. This tries several free-tier models and uses the first that works.
CANDIDATE_MODELS = ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-3.5-flash"]
def pick_model():
    for m in CANDIDATE_MODELS:
        try:
            client.models.generate_content(model=m, contents="ping")
            print("Using model:", m)
            return m
        except Exception as e:
            print("Skipping", m, "->", str(e)[:100])
    raise RuntimeError("No model worked. Check your key, account and the model list at https://ai.google.dev/gemini-api/docs/models")
MODEL = pick_model()

## Step 1 · A tool is just a Python function
The docstring tells the model when and how to use it.

In [ ]:
CROP_TIPS = {
    ("maize", "yellow leaves"): "Check nitrogen, drainage and stalk borer. Apply manure or fertiliser as advised locally.",
    ("maize", "holes in leaves"): "Possible fall armyworm. Inspect whorls early in the morning and contact the extension officer.",
    ("beans", "yellow leaves"): "Check waterlogging and leaf diseases. Remove badly affected plants.",
}

def crop_advice(crop: str, problem: str) -> dict:
    """Look up advice for a crop problem.

    Args:
        crop: crop name in English, for example 'maize' or 'beans'.
        problem: short problem description in English, for example 'yellow leaves'.
    """
    key = (crop.lower().strip(), problem.lower().strip())
    return {"advice": CROP_TIPS.get(key, "No record. Ask the local extension officer.")}

print(crop_advice("maize", "yellow leaves"))

## Step 2 · Give the tool to Gemini
The SDK calls your function automatically when the model asks for it.

In [ ]:
config = types.GenerateContentConfig(
    system_instruction="You advise smallholder farmers briefly, in the farmer's language (Swahili by default). Use crop_advice for crop problems. If there is no record, say you are not sure.",
    tools=[crop_advice],
)
r = client.models.generate_content(model=MODEL, contents="Majani ya mahindi yangu yanageuka manjano. Nifanye nini?", config=config)
print(r.text)

## Step 3 · A short conversation (the agent remembers the chat)

In [ ]:
chat = client.chats.create(model=MODEL, config=config)
for msg in ["Habari! Mahindi yangu yana mashimo kwenye majani.", "Nikiona wadudu asubuhi, nifanye nini kwanza?"]:
    print("Farmer:", msg)
    print("Agent :", chat.send_message(msg).text, "\n")

## Your turn
- Add two more rows to `CROP_TIPS` for crops grown near you.
- Ask about a crop that is *not* in the table. Does the agent admit it is unsure?